## 3. A model of challenge inventory

**Setup.** One team faces `T = 54` opportunities (six per inning, a modeling choice). On each opportunity a called pitch arises with probability `r`, and its true overturn probability `q` is drawn from a Beta distribution with mean `mu` (concentration fixed at 4). The team holds `k` challenges with `t` opportunities left. The value of a flipped call is `v` (one in the baseline).

`W(k,t) = (1-r) W(k,t-1) + r E_q[ max( W(k,t-1),  q (v + W(k,t-1)) + (1-q) W(k-1,t-1) ) ]`

The option value of one challenge is `D(k,t) = W(k,t-1) - W(k-1,t-1)`, and challenging beats waiting exactly when `q >= D / (v + D)`. As the game nears its end, `D` falls toward zero, so the threshold falls. If `v` rises late, the threshold falls faster. The same code covers both cases through a stakes multiplier `m`: stakes rise linearly from 1 in the first inning to `m` in the ninth.

**Externality.** Nine batters and a catcher share one pool. A batter perceives only a share `phi` of the inventory cost, so his threshold is `phi*D / (v + phi*D)`, lower than the team's. The defense bears the full cost. A batter initiates each opportunity with probability 0.451, the observed batter share of challenges.

The model ends at the ninth inning and ignores the extra-innings rule.

In [ ]:
from abs_common import *        

In [ ]:
S, T, NQ, KAPPA = 6, 54, 200, 4.0       # opportunities per inning, total, q grid, Beta concentration
QS = (np.arange(NQ) + .5) / NQ
SHARE_B = 0.451                         # observed batter share of challenges (4,766 / 10,557)

def q_weights(mu):
    a, b = mu*KAPPA, (1-mu)*KAPPA
    w = QS**(a-1) * (1-QS)**(b-1)
    return w / w.sum()

def stakes(t, m):
    """Stakes of a flipped call with t opportunities left: 1 in inning 1, rising linearly to m in inning 9."""
    return 1 + (m-1) * ((T-t)//S) / 8

def solve(r, mu, m=1.0):
    """Backward induction. Returns W (value) and D (option value of one challenge), indexed [k, t]."""
    w = q_weights(mu)
    W = np.zeros((3, T+1)); D = np.zeros((3, T+1))
    for t in range(1, T+1):
        v = stakes(t, m)
        W[0, t] = W[0, t-1]
        for k in (1, 2):
            d = W[k, t-1] - W[k-1, t-1]; D[k, t] = d
            th = d / (v + d)
            use = QS*(v + W[k, t-1]) + (1-QS)*W[k-1, t-1]
            W[k, t] = (1-r)*W[k, t-1] + r*(w*np.where(QS >= th, use, W[k, t-1])).sum()
    return W, D

def simulate(r, mu, m=1.0, phi_b=1.0, N=60_000, rng=None, draw_b=True):
    """Simulate N games. Returns one record per challenge made plus per-game challenge counts.
    Random draws are made in a fixed order (arrival, q, batter flag, outcome) so a seeded stream replays exactly."""
    rng = rng if rng is not None else np.random.default_rng(SEED_FIT)
    cum = np.cumsum(q_weights(mu)); cum /= cum[-1]; W, D = solve(r, mu, m)
    k = np.full(N, 2); n_game = np.zeros(N, int); ev = {n: [] for n in ("inn","k","ok","is_b","v")}
    for s in range(T):
        t, i = T - s, s//S + 1; v = stakes(t, m)
        arrive = rng.random(N) < r
        q = QS[np.minimum(np.searchsorted(cum, rng.random(N), side="right"), NQ-1)]
        is_b = rng.random(N) < SHARE_B if draw_b else np.zeros(N, bool)
        d = D[k, t]; f = np.where(is_b, phi_b, 1.0)
        ch = arrive & (k > 0) & (q >= f*d/(v + f*d))
        ok = ch & (rng.random(N) < q)
        n_game += ch
        for name, arr in (("inn", np.full(ch.sum(), i)), ("k", k[ch]), ("ok", ok[ch]),
                          ("is_b", is_b[ch]), ("v", np.full(ch.sum(), v))):
            ev[name].append(arr)
        k = k - (ch & ~ok)
    out = {n: np.concatenate(a) for n, a in ev.items()}
    out["n_game"], out["N"] = n_game, N
    return out

def summarize(s):
    return dict(chal_per_game=len(s["ok"])/s["N"], rate=s["ok"].mean(), zero=(s["n_game"] == 0).mean())

# Calibration targets, as stated in the paper: challenges per team-game, overall overturn rate, share of team-games with none
TGT = dict(chal_per_game=2.18, rate=0.539, zero=0.083)
_CACHE = {}

def game_model_run():
    """Replay the paper's main simulation stream (seed 7) in its original order: the grid fit, the base run,
    the batter-externality runs (phi = 1, 1/9, 0), the run behind Figure 2, then extra phi values."""
    if "game" in _CACHE: return _CACHE["game"]
    rng = np.random.default_rng(SEED_FIT); grid = []
    for r, mu in itertools.product([.03,.04,.05,.06,.08,.10], [.35,.45,.55,.65,.75]):
        m = summarize(simulate(r, mu, N=20_000, rng=rng))
        err = ((m["chal_per_game"]-TGT["chal_per_game"])/TGT["chal_per_game"])**2 \
            + ((m["rate"]-TGT["rate"])/TGT["rate"])**2 + ((m["zero"]-TGT["zero"])/.05)**2
        grid.append(dict(r=r, mu=mu, err=err, **m))
    grid = pd.DataFrame(grid).sort_values("err")
    r, mu = float(grid.iloc[0].r), float(grid.iloc[0].mu)
    run = dict(grid=grid, r=r, mu=mu, base=simulate(r, mu, N=60_000, rng=rng))
    for phi in (1.0, 1/9, 0.0): run[phi] = simulate(r, mu, phi_b=phi, N=60_000, rng=rng)
    run["fig"] = simulate(r, mu, N=60_000, rng=rng)
    for phi in (0.5, 0.25): run[phi] = simulate(r, mu, phi_b=phi, N=60_000, rng=rng)
    _CACHE["game"] = run
    return run

def timing_run(ms=(1.0, 1.5, 2.0, 3.0), r=0.05, mu=0.45, N=80_000):
    """Replay the timing extension's own stream (seed 3): one run per stakes multiplier, no batter draw."""
    if "timing" in _CACHE: return _CACHE["timing"]
    rng = np.random.default_rng(SEED_TIMING)
    _CACHE["timing"] = {m: simulate(r, mu, m=m, N=N, rng=rng, draw_b=False) for m in ms}
    return _CACHE["timing"]